<a href="https://colab.research.google.com/github/rafayraza-nextgen/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rafayraza-nextgen/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

We rank pages based on their potential for fast growth. The top priority is "Quick Win - Update Title" for pages with high views but a low Click-Through Rate (CTR). The second priority is "Protect - Monitor Traffic" for pages that are already doing well. We ignore pages with very low traffic.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd
from datasets import load_dataset
from google.colab import userdata

print("Loading data to build the final playbook...")
hf_token = userdata.get('HF_TOKEN')
dataset = load_dataset("FlyRank/internship-warehouse", "fact_content_query_90d", token=hf_token)
# Load a sample to build the queue quickly
df = dataset['train'].to_pandas().head(50000).copy()

# Create the action rules
df['ctr'] = (df['clicks_90d'] / df['impressions_90d']).fillna(0)
df['action'] = 'Ignore - Low Traffic'

# Apply the rules we learned from the model
df.loc[(df['impressions_90d'] > 1000) & (df['ctr'] < 0.02), 'action'] = 'Quick Win - Update Title'
df.loc[(df['impressions_90d'] > 1000) & (df['ctr'] >= 0.02), 'action'] = 'Protect - Monitor Traffic'

# Filter out the ignored pages and sort by impressions
playbook = df[df['action'] != 'Ignore - Low Traffic'].copy()
playbook = playbook.sort_values(by='impressions_90d', ascending=False)

print("Top 5 actions in the queue:")
display_cols = ['client_hash_id', 'impressions_90d', 'clicks_90d', 'action']
display(playbook[display_cols].head())


Loading data to build the final playbook...


README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

fact_content_query_90d.parquet: reconstructing file:   0%|          |  0.00B / 60.7MB            

fact_content_query_90d.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2414248 [00:00<?, ? examples/s]

Top 5 actions in the queue:


,client_hash_id,impressions_90d,clicks_90d,action
31948,client_20259bd6705d81d4,54353,121,Quick Win - Update Title
13135,client_0fa64a184f18a4a0,39819,9,Quick Win - Update Title
10676,client_0fa64a184f18a4a0,33720,15,Quick Win - Update Title
42285,client_23a62021009f63c4,24082,31,Quick Win - Update Title
32901,client_20259bd6705d81d4,20638,63,Quick Win - Update Title


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

This tool is made for SEO managers and content teams. It helps them decide which web pages to fix first to save time. It is a decision-support tool, not an automatic editor. It stops being valid for brand new pages because they do not have enough traffic history yet.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("--- INTENDED USE ---")
print("Target Users: Content Teams, SEO Managers, Writers.")
print("Goal: Prioritize which pages need human review first.")
print("\n--- LIMITATIONS ---")
print("Data Limits: This model cannot score brand new pages (under 90 days old).")
print("Action Limits: This model only suggests actions, it does not rewrite text.")

--- INTENDED USE ---
Target Users: Content Teams, SEO Managers, Writers.
Goal: Prioritize which pages need human review first.

--- LIMITATIONS ---
Data Limits: This model cannot score brand new pages (under 90 days old).
Action Limits: This model only suggests actions, it does not rewrite text.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

A human must always read the web page before making any changes. The model might flag a page as "low clicks," but the page might just be a simple contact form. We should never automate updates to legal pages, pricing details, or company contact pages.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

no_go_list = [
    'Legal Terms and Conditions',
    'Privacy Policies',
    'Checkout and Payment Pages',
    'Contact Us / Phone Number Pages'
]

print("THE NO-GO LIST (Never Automate Updates Here):")
for item in no_go_list:
    print(f"- {item}")


THE NO-GO LIST (Never Automate Updates Here):
- Legal Terms and Conditions
- Privacy Policies
- Checkout and Payment Pages
- Contact Us / Phone Number Pages


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The web changes every day. The model will go stale if Google releases a major search algorithm update. We must monitor the system and retrain the model if it has been more than 90 days, or if the click-through rates suddenly drop across all pages.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Define our safety triggers
days_since_training = 95
google_core_update_happened = False

print("Checking model health...")

if days_since_training > 90 or google_core_update_happened:
    print("ALERT: Retrain Trigger Hit! The data is stale or the search rules changed.")
    print("Action Required: Retrain the clustering model with fresh data.")
else:
    print("Status: Model is healthy and up to date.")


Checking model health...
ALERT: Retrain Trigger Hit! The data is stale or the search rules changed.
Action Required: Retrain the clustering model with fresh data.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

I am saving this final ranked action list to a CSV file in the outputs folder. I will use this exported file to write my final Capstone Research Paper.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os

# Create the outputs folder if it does not exist
os.makedirs('work/outputs', exist_ok=True)

# Save the final playbook
export_path = 'work/outputs/final_action_playbook.csv'
playbook[display_cols].to_csv(export_path, index=False)

print(f"Success! The final playbook is saved to: {export_path}")

Success! The final playbook is saved to: work/outputs/final_action_playbook.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.